# Base Year QA — 2022 vs 2026 Housing EIS

Compares the four model input CSVs produced by the 2026 base year data engineering pipeline against the equivalent 2022 RTP base year outputs.

**Files compared:**
- `SchoolEnrollment.csv`
- `SocioEcon_Summer.csv`
- `VisitorOccupancyRates_Summer.csv`
- `OvernightVisitorZonalData_Summer.csv`

**2022 source:** `TravelDemandModel/2022/data/processed_data/`  
**2026 source:** `TravelDemandModel/2026_Housing_EIS/Base/data/processed_data/`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

pd.options.display.max_columns = 30
pd.options.display.max_rows    = 60
pd.options.display.float_format = '{:,.2f}'.format

REPO_ROOT = Path().absolute().parents[3]   # adjust if running from a different cwd
DIR_22 = REPO_ROOT / 'TravelDemandModel/2022/data/processed_data'
DIR_26 = REPO_ROOT / 'TravelDemandModel/2026_Housing_EIS/Base/data/processed_data'

# Verify paths
for p in [DIR_22, DIR_26]:
    assert p.exists(), f'Path not found: {p}'

print('2022 dir:', DIR_22)
print('2026 dir:', DIR_26)

In [ ]:
# ── Helper functions ─────────────────────────────────────────────────────────

def load_both(filename, taz_col=None):
    """Load a CSV from both years; normalise the TAZ column to int and set as index."""
    df22 = pd.read_csv(DIR_22 / filename)
    df26 = pd.read_csv(DIR_26 / filename)
    # Detect TAZ column automatically if not provided
    if taz_col is None:
        taz_col = 'TAZ' if 'TAZ' in df22.columns else 'taz'
    df22[taz_col] = pd.to_numeric(df22[taz_col], errors='coerce').astype('Int64')
    df26[taz_col] = pd.to_numeric(df26[taz_col], errors='coerce').astype('Int64')
    df22 = df22.set_index(taz_col)
    df26 = df26.set_index(taz_col)
    return df22, df26


def basin_totals(df22, df26):
    """Return a DataFrame with column sums for both years and absolute/pct difference."""
    num_cols = [c for c in df22.columns if c in df26.columns]
    s22 = df22[num_cols].sum()
    s26 = df26[num_cols].sum()
    diff = s26 - s22
    pct  = (diff / s22.replace(0, np.nan) * 100).round(1)
    return pd.DataFrame({'2022': s22, '2026': s26, 'diff': diff, 'pct_change': pct})


def per_taz_diff(df22, df26, cols=None):
    """Align on TAZ index and return a diff DataFrame (2026 - 2022) for shared TAZs."""
    shared = df22.index.intersection(df26.index)
    num_cols = cols or [c for c in df22.columns if c in df26.columns]
    diff = df26.loc[shared, num_cols].subtract(df22.loc[shared, num_cols])
    return diff


def flag_large_changes(diff_df, threshold=50):
    """Print TAZs where any column changed by more than threshold."""
    mask = diff_df.abs().max(axis=1) > threshold
    flagged = diff_df[mask]
    if flagged.empty:
        print(f'  No TAZ with change > {threshold}.')
    else:
        print(f'  {len(flagged)} TAZ(s) with change > {threshold}:')
        display(flagged[flagged.abs().max(axis=1) > threshold].sort_index())


def scatter_grid(df22, df26, title, cols=None, figsize=(14, 4)):
    """2022 vs 2026 scatter for each numeric column on shared TAZs."""
    shared = df22.index.intersection(df26.index)
    num_cols = cols or [c for c in df22.columns if c in df26.columns]
    ncols = len(num_cols)
    if ncols == 0:
        return
    fig, axes = plt.subplots(1, ncols, figsize=(figsize[0], figsize[1]))
    if ncols == 1:
        axes = [axes]
    fig.suptitle(title, fontsize=12, fontweight='bold')
    for ax, col in zip(axes, num_cols):
        x = df22.loc[shared, col].fillna(0)
        y = df26.loc[shared, col].fillna(0)
        ax.scatter(x, y, alpha=0.4, s=10, color='steelblue')
        lim = max(x.max(), y.max()) * 1.05
        ax.plot([0, lim], [0, lim], 'r--', linewidth=0.8, label='1:1 line')
        ax.set_xlabel('2022', fontsize=9)
        ax.set_ylabel('2026', fontsize=9)
        ax.set_title(col, fontsize=9)
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
    plt.tight_layout()
    plt.show()


def diff_bar(diff_df, title, top_n=20, figsize=(12, 4)):
    """Bar chart of per-TAZ differences, showing the top_n absolute movers."""
    net = diff_df.sum(axis=1).sort_values(key=abs, ascending=False).head(top_n)
    colors = ['crimson' if v < 0 else 'steelblue' for v in net]
    fig, ax = plt.subplots(figsize=figsize)
    ax.bar(net.index.astype(str), net.values, color=colors)
    ax.axhline(0, color='black', linewidth=0.7)
    ax.set_title(f'{title} — top {top_n} TAZ movers (net diff)', fontsize=11)
    ax.set_xlabel('TAZ')
    ax.set_ylabel('2026 − 2022')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.tight_layout()
    plt.show()

---
## 1. SchoolEnrollment.csv

In [ ]:
se22, se26 = load_both('SchoolEnrollment.csv', taz_col='TAZ')

print(f'2022 TAZ count : {len(se22)}')
print(f'2026 TAZ count : {len(se26)}')

new_tazs = se26.index.difference(se22.index).tolist()
removed_tazs = se22.index.difference(se26.index).tolist()
print(f'TAZs in 2026 only (phantom): {sorted(new_tazs)}')
print(f'TAZs in 2022 only          : {sorted(removed_tazs)}')

In [ ]:
# Phantom TAZs — confirm all zeros
if new_tazs:
    print('Phantom TAZ values:')
    display(se26.loc[sorted(new_tazs)])

In [ ]:
print('Basin totals (shared TAZs only):')
display(basin_totals(se22, se26))

In [ ]:
se_diff = per_taz_diff(se22, se26)
print('TAZs with school enrollment changes:')
changed = se_diff[(se_diff != 0).any(axis=1)]
if changed.empty:
    print('  None — enrollment identical across shared TAZs.')
else:
    display(changed.sort_index())

In [ ]:
scatter_grid(se22, se26, 'SchoolEnrollment — 2022 vs 2026 (per TAZ)',
             cols=['elementary_school_enrollment','middle_school_enrollment',
                   'high_school_enrollment','college_enrollment'], figsize=(16, 4))

---
## 2. SocioEcon_Summer.csv

In [ ]:
se22s, se26s = load_both('SocioEcon_Summer.csv', taz_col='taz')

print(f'2022 TAZ count : {len(se22s)}')
print(f'2026 TAZ count : {len(se26s)}')

print('\nBasin totals:')
display(basin_totals(se22s, se26s))

In [ ]:
# Residential and population columns
res_cols = ['total_residential_units','total_occ_units','total_persons',
            'occ_units_low_inc','occ_units_med_inc','occ_units_high_inc']
scatter_grid(se22s, se26s, 'SocioEcon — residential & population', cols=res_cols, figsize=(18, 4))

In [ ]:
# Employment columns (were zero in 2022, now populated in 2026)
emp_cols = ['emp_retail','emp_srvc','emp_rec','emp_game','emp_other']
print('Employment basin totals (note: 2022 run had placeholder zeros):')
display(basin_totals(se22s[emp_cols], se26s[emp_cols]))

In [ ]:
# Per-TAZ diff — residential and pop only
soc_diff = per_taz_diff(se22s, se26s, cols=res_cols + ['persons_per_occ_unit'])
print('Largest TAZ-level movers (residential units + persons):')
flag_large_changes(soc_diff[['total_residential_units','total_occ_units','total_persons']], threshold=100)

diff_bar(soc_diff[['total_residential_units','total_occ_units','total_persons']],
         'SocioEcon — residential / population net diff')

In [ ]:
# Occupancy rate comparison (rate column — should be similar)
plt.figure(figsize=(8, 4))
shared = se22s.index.intersection(se26s.index)
plt.scatter(se22s.loc[shared,'census_occ_rate'], se26s.loc[shared,'census_occ_rate'],
            alpha=0.4, s=10, color='steelblue')
mx = max(se22s.loc[shared,'census_occ_rate'].max(), se26s.loc[shared,'census_occ_rate'].max()) * 1.05
plt.plot([0, mx], [0, mx], 'r--', linewidth=0.8)
plt.xlabel('2022 census_occ_rate'); plt.ylabel('2026 census_occ_rate')
plt.title('Census Occupancy Rate — 2022 vs 2026 (per TAZ)')
plt.tight_layout(); plt.show()

---
## 3. VisitorOccupancyRates_Summer.csv

In [ ]:
vo22, vo26 = load_both('VisitorOccupancyRates_Summer.csv', taz_col='taz')

print(f'2022 TAZ count : {len(vo22)}')
print(f'2026 TAZ count : {len(vo26)}')

print('\nBasin sums of rates (informational — rates are per-TAZ, not additive):')
display(basin_totals(vo22, vo26))

In [ ]:
# TAZ-level rate comparison
rate_cols = ['hotelmotel','resort','casino','campground','house','seasonal']
scatter_grid(vo22, vo26, 'VisitorOccupancyRates — 2022 vs 2026 (per TAZ)', cols=rate_cols, figsize=(18, 4))

In [ ]:
# Flag TAZs where any rate changed materially (>0.05)
vo_diff = per_taz_diff(vo22, vo26)
large_rate_changes = vo_diff[vo_diff.abs().max(axis=1) > 0.05]
if large_rate_changes.empty:
    print('No TAZ with occupancy rate change > 0.05.')
else:
    print(f'{len(large_rate_changes)} TAZ(s) with occupancy rate change > 0.05:')
    display(large_rate_changes.sort_index())

In [ ]:
# Campground occupancy rates — should be identical (2026 copies from example file)
shared = vo22.index.intersection(vo26.index)
camp_match = (vo22.loc[shared,'campground'].round(6) == vo26.loc[shared,'campground'].round(6)).all()
print(f'Campground occupancy rates identical across shared TAZs: {camp_match}')

house_diff = (vo26.loc[shared,'house'] - vo22.loc[shared,'house']).describe()
print('\nHouse rate diff (2026-2022) stats:')
display(house_diff.to_frame('diff'))

---
## 4. OvernightVisitorZonalData_Summer.csv

In [ ]:
ov22, ov26 = load_both('OvernightVisitorZonalData_Summer.csv', taz_col='taz')

print(f'2022 TAZ count : {len(ov22)}')
print(f'2026 TAZ count : {len(ov26)}')
print(f'2026 new columns: {[c for c in ov26.columns if c not in ov22.columns]}')

print('\nBasin totals:')
display(basin_totals(ov22, ov26))

In [ ]:
# TAZ-level scatter for lodging + campground
lodging_cols = ['hotelmotel','resort','casino','campground']
scatter_grid(ov22, ov26, 'OvernightVisitorZonalData — lodging & campground', cols=lodging_cols, figsize=(16, 4))

In [ ]:
# Flag TAUs with large changes
ov_diff = per_taz_diff(ov22, ov26, cols=lodging_cols)
print('TAZs with lodging unit changes > 10:')
flag_large_changes(ov_diff, threshold=10)

In [ ]:
diff_bar(ov_diff, 'OvernightVisitorZonalData — lodging net diff per TAZ')

In [ ]:
# percentHouseSeasonal comparison
shared = ov22.index.intersection(ov26.index)
plt.figure(figsize=(7, 4))
plt.scatter(ov22.loc[shared,'percentHouseSeasonal'], ov26.loc[shared,'percentHouseSeasonal'],
            alpha=0.4, s=10, color='steelblue')
mx = max(ov22.loc[shared,'percentHouseSeasonal'].max(), ov26.loc[shared,'percentHouseSeasonal'].max()) * 1.05
plt.plot([0, mx], [0, mx], 'r--', linewidth=0.8)
plt.xlabel('2022'); plt.ylabel('2026')
plt.title('percentHouseSeasonal — 2022 vs 2026 (per TAZ)')
plt.tight_layout(); plt.show()

phs_diff = ov26.loc[shared,'percentHouseSeasonal'] - ov22.loc[shared,'percentHouseSeasonal']
print('percentHouseSeasonal diff stats:')
display(phs_diff.describe().to_frame('diff'))

In [ ]:
# Campground detail — by TAZ
camp22 = ov22[ov22['campground'] > 0]['campground']
camp26 = ov26[ov26['campground'] > 0]['campground']
camp_df = pd.DataFrame({'2022': camp22, '2026': camp26}).fillna(0).astype(int)
camp_df['diff'] = camp_df['2026'] - camp_df['2022']
print('Campground sites by TAZ (non-zero in either year):')
display(camp_df.sort_index())
print(f'\nBasin total 2022: {camp_df["2022"].sum():,}')
print(f'Basin total 2026: {camp_df["2026"].sum():,}')
print(f'Difference      : {camp_df["diff"].sum():+,}')

---
## 5. Summary Dashboard

In [ ]:
# Key metrics comparison table
metrics = {
    'Residential units (total)':          (se22s['total_residential_units'].sum(),  se26s['total_residential_units'].sum()),
    'Occupied units (total)':             (se22s['total_occ_units'].sum(),           se26s['total_occ_units'].sum()),
    'Total persons':                      (se22s['total_persons'].sum(),             se26s['total_persons'].sum()),
    'School enrollment (all types)':      (se22['elementary_school_enrollment'].sum() +
                                           se22['middle_school_enrollment'].sum() +
                                           se22['high_school_enrollment'].sum() +
                                           se22['college_enrollment'].sum(),
                                           se26['elementary_school_enrollment'].sum() +
                                           se26['middle_school_enrollment'].sum() +
                                           se26['high_school_enrollment'].sum() +
                                           se26['college_enrollment'].sum()),
    'TAU rooms — HotelMotel':             (ov22['hotelmotel'].sum(),                 ov26['hotelmotel'].sum()),
    'TAU rooms — Resort':                 (ov22['resort'].sum(),                     ov26['resort'].sum()),
    'TAU rooms — Casino':                 (ov22['casino'].sum(),                     ov26['casino'].sum()),
    'Campground sites':                   (ov22['campground'].sum(),                 ov26['campground'].sum()),
    'Employment (total)':                 (se22s[['emp_retail','emp_srvc','emp_rec','emp_game','emp_other']].sum().sum(),
                                           se26s[['emp_retail','emp_srvc','emp_rec','emp_game','emp_other']].sum().sum()),
}

summary = pd.DataFrame(
    [(k, f'{v22:,.0f}', f'{v26:,.0f}', f'{v26-v22:+,.0f}', f'{(v26-v22)/max(v22,1)*100:+.1f}%')
     for k, (v22, v26) in metrics.items()],
    columns=['Metric','2022','2026','Diff','Pct Change']
).set_index('Metric')

display(summary)

In [ ]:
# TAZ coverage check
print('TAZ counts:')
print(f'  SchoolEnrollment : 2022={len(se22)}, 2026={len(se26)}')
print(f'  SocioEcon        : 2022={len(se22s)}, 2026={len(se26s)}')
print(f'  VisitorOccRates  : 2022={len(vo22)}, 2026={len(vo26)}')
print(f'  OvernightVisitor : 2022={len(ov22)}, 2026={len(ov26)}')

print('\nPhantom TAZs present in 2026 SchoolEnrollment (all zeros):')
print(' ', sorted(se26.index.difference(se22.index).tolist()))
print('  These are low-use/empty zones added for model completeness.')